# Processes, Monitoring, and Job Control

We have explored how to create programs, make them executable, and how to make sure our user has the permission to run them. 

One running instance of a program is a __process__.  

A long-running background process that provides a service is a __daemon__. 

This notebook introduces tools to monitor running processes, and perform job control.

## Start, inspect, and stop a controlled process

The shell waits for a command unless you add `&` at the end, which starts it in the background. 

For example, start a timer, save its process identifier (PID), inspect only that PID, and stop only that timer:

```bash
sleep 30 &
SLEEP_PID=$!
jobs -l
ps -p "$SLEEP_PID" -o pid=,stat=,command=
kill "$SLEEP_PID"
wait "$SLEEP_PID" || printf '%s\n' 'The timer was stopped.'
jobs -l
```

`$!` contains the PID of the most recent background process. `jobs -l` lists background jobs started by the current shell and their PIDs. `ps` shows information for the saved PID. `kill "$SLEEP_PID"` sends `SIGTERM`, the default termination signal. A process may handle, ignore, or terminate in response, so it is a request rather than a guarantee of a clean shutdown. Never run `kill` with an arbitrary number: identify the process first and use it only for the learner-owned timer here.

`wait "$SLEEP_PID"` pauses until that exact process finishes and returns its exit status. Because the timer was deliberately stopped, the `||` message is expected. The final `jobs -l` should no longer list it. A process occupying the terminal can usually be interrupted with `Ctrl-C`; a background process continues while the shell accepts another command.

To bring a job started by this shell back to the foreground, use its bracketed number from your own `jobs -l` output. The example uses `%1`; replace it with the number shown in your output. For example, if `jobs -l` shows `[2]`, run `fg %2`:

```bash
sleep 30 &
jobs -l
fg %1
```

`fg` waits for that timer to finish; it does not select another user's process.

## Monitor the correct shell context

When your local Linux environment provides `htop`, use `htop --readonly`, an interactive process monitor that refreshes process, CPU, memory, and swap information without allowing process-changing actions:

```bash
htop --readonly
```

Run it in the same shell context as the system you intend to observe. An isolated development environment shows its own processes and limits: an ordinary base-station terminal shows base-station processes. A shell in a service-specific environment may show a restricted process view when that environment uses a separate PID namespace or a restrictive `/proc` configuration. A high CPU or memory value is evidence to investigate, not a diagnosis. Press `q` or `F10` to leave.

Do not use signal or kill actions in a monitor. On a system where `htop` is unavailable, use `top`, press `q` to leave, and do not install packages or use elevated commands just to reproduce this display. Monitor only systems and service environments you own or are explicitly authorized to administer.

## Further reading

The Bash Reference Manual explains [job control](https://www.gnu.org/software/bash/manual/html_node/Job-Control-Basics.html).

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
